# Sleep Efficiency – Predicting Sleep Quality with a Neural Network

**Group project – ANN course**

In this project we predict how efficiently a person sleeps based on lifestyle factors such as caffeine intake, alcohol consumption, exercise frequency, and smoking habits.

Sleep efficiency is defined as the ratio of actual sleep time to total time spent in bed (0–1 scale).

**Dataset:** Sleep Efficiency Dataset (Kaggle)  
**Target variable:** `Sleep efficiency` (continuous, regression problem)  
**Model:** Artificial Neural Network (ANN) using Keras / TensorFlow

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Make plots look a bit nicer
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('TensorFlow version:', tf.__version__)

---
## 2. Load the Dataset

In [ ]:
df = pd.read_csv('data/Sleep_Efficiency.csv')

print('Shape:', df.shape)
df.head()

In [ ]:
# Quick overview of data types and missing values
df.info()

In [ ]:
# Basic statistics for all numerical columns
df.describe()

---
## 3. Data Cleaning

Before we can analyze or model the data, we need to clean it up.
This includes handling missing values, fixing data types, and encoding categorical variables.

In [ ]:
# Check how many missing values each column has
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Visualize missing values
missing = df.isnull().sum()
missing = missing[missing > 0]

plt.figure(figsize=(8, 4))
missing.plot(kind='bar', color='steelblue')
plt.title('Number of Missing Values per Column')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Fill missing values with the median for numerical columns
# We use median instead of mean because it is less sensitive to outliers
cols_to_fill = ['Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Exercise frequency']

for col in cols_to_fill:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'{col}: filled with median = {median_val}')

print('\nMissing values after cleaning:')
print(df.isnull().sum())

In [ ]:
# Drop columns we don't need for the model
# ID is just a row number, Bedtime and Wakeup time are already captured in Sleep duration
df = df.drop(columns=['ID', 'Bedtime', 'Wakeup time'])

print('Columns after dropping:', df.columns.tolist())

In [ ]:
# Encode categorical variables
# Gender: Female = 0, Male = 1
df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})

# Smoking status: No = 0, Yes = 1
df['Smoking status'] = df['Smoking status'].map({'No': 0, 'Yes': 1})

print('Encoding done!')
df.head()

In [ ]:
# Check for duplicate rows
print('Duplicate rows:', df.duplicated().sum())

---
## 4. Exploratory Data Analysis (EDA)

Now that the data is clean, we want to explore it visually to understand patterns and relationships.
This is important for understanding what drives sleep efficiency.

### 4.1 Distribution of Sleep Efficiency (our target variable)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['Sleep efficiency'], bins=20, color='steelblue', edgecolor='white')
plt.title('Distribution of Sleep Efficiency')
plt.xlabel('Sleep Efficiency')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.boxplot(df['Sleep efficiency'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.7))
plt.title('Boxplot of Sleep Efficiency')
plt.xlabel('Sleep Efficiency')

plt.tight_layout()
plt.show()

print(f"Mean sleep efficiency: {df['Sleep efficiency'].mean():.2f}")
print(f"Median sleep efficiency: {df['Sleep efficiency'].median():.2f}")
print(f"Std deviation: {df['Sleep efficiency'].std():.2f}")

### 4.2 Distribution of Key Variables

In [ ]:
cols_to_plot = ['Age', 'Sleep duration', 'Awakenings', 'Caffeine consumption',
                'Alcohol consumption', 'Exercise frequency']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    axes[i].hist(df[col], bins=15, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Sleep Efficiency vs Lifestyle Factors

In [ ]:
# How does caffeine affect sleep efficiency?
plt.figure(figsize=(8, 5))
sns.boxplot(x='Caffeine consumption', y='Sleep efficiency', data=df, palette='Blues')
plt.title('Sleep Efficiency by Caffeine Consumption (mg)')
plt.xlabel('Caffeine Consumption (mg)')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

In [ ]:
# How does alcohol affect sleep efficiency?
plt.figure(figsize=(8, 5))
sns.boxplot(x='Alcohol consumption', y='Sleep efficiency', data=df, palette='Oranges')
plt.title('Sleep Efficiency by Alcohol Consumption')
plt.xlabel('Alcohol Consumption (units)')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

In [ ]:
# How does exercise frequency affect sleep efficiency?
plt.figure(figsize=(8, 5))
sns.boxplot(x='Exercise frequency', y='Sleep efficiency', data=df, palette='Greens')
plt.title('Sleep Efficiency by Exercise Frequency')
plt.xlabel('Exercise Frequency (days/week)')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

In [ ]:
# Does smoking affect sleep efficiency?
plt.figure(figsize=(6, 5))
sns.boxplot(x='Smoking status', y='Sleep efficiency', data=df, palette='Set2')
plt.xticks([0, 1], ['Non-smoker', 'Smoker'])
plt.title('Sleep Efficiency: Smokers vs Non-smokers')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot: Sleep duration vs Sleep efficiency
plt.figure(figsize=(8, 5))
plt.scatter(df['Sleep duration'], df['Sleep efficiency'], alpha=0.5, color='steelblue')
plt.title('Sleep Duration vs Sleep Efficiency')
plt.xlabel('Sleep Duration (hours)')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot: Age vs Sleep efficiency
plt.figure(figsize=(8, 5))
plt.scatter(df['Age'], df['Sleep efficiency'], alpha=0.5, color='coral')
plt.title('Age vs Sleep Efficiency')
plt.xlabel('Age')
plt.ylabel('Sleep Efficiency')
plt.tight_layout()
plt.show()

### 4.4 Correlation Matrix

In [ ]:
# Correlation matrix shows how strongly variables are related to each other
plt.figure(figsize=(12, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Which variables correlate most with sleep efficiency?
corr_with_target = corr['Sleep efficiency'].drop('Sleep efficiency').sort_values(ascending=False)

plt.figure(figsize=(8, 5))
corr_with_target.plot(kind='bar', color=['green' if x > 0 else 'red' for x in corr_with_target])
plt.title('Correlation with Sleep Efficiency')
plt.ylabel('Correlation coefficient')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print(corr_with_target)

---
## 5. Prepare Data for the Model

In [ ]:
# Define features (X) and target (y)
# We drop the sleep percentage columns since they are consequences of sleep efficiency,
# not causes – keeping them would make the model "cheat"
X = df.drop(columns=['Sleep efficiency', 'REM sleep percentage',
                      'Deep sleep percentage', 'Light sleep percentage'])
y = df['Sleep efficiency']

print('Features:', X.columns.tolist())
print('Target: Sleep efficiency')
print('Shape X:', X.shape)
print('Shape y:', y.shape)

In [ ]:
# Split into training set (80%) and test set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])

In [ ]:
# Scale the features using StandardScaler
# This ensures all variables are on the same scale, which helps the neural network train better
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # fit on training data only
X_test_scaled = scaler.transform(X_test)         # transform test data with same scaler

print('Scaling done!')

---
## 6. Build the Neural Network Model

We build a simple Sequential model with two hidden layers.
Since this is a regression problem, the output layer has one neuron with no activation function.

In [ ]:
# Set a random seed for reproducibility
tf.random.set_seed(42)

# Build the model
model = keras.Sequential([
    # Input layer + first hidden layer
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),

    # Second hidden layer
    layers.Dense(32, activation='relu'),

    # Output layer – one neuron for regression (no activation function)
    layers.Dense(1)
])

model.summary()

In [ ]:
# Compile the model
# Adam optimizer adjusts the learning rate automatically
# MSE (Mean Squared Error) is the standard loss function for regression
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']  # Mean Absolute Error – easier to interpret
)

print('Model compiled!')

---
## 7. Train the Model

In [ ]:
# Train the model
# We use 20% of training data as validation to monitor overfitting
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

---
## 8. Evaluate the Model

In [ ]:
# Plot training history – Loss over epochs
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('MAE over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test data
y_pred = model.predict(X_test_scaled).flatten()

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('--- Model Performance on Test Data ---')
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R²:   {r2:.4f}')

In [ ]:
# Plot: Actual vs Predicted sleep efficiency
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='red', linestyle='--', linewidth=1.5, label='Perfect prediction')
plt.title('Actual vs Predicted Sleep Efficiency')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of prediction errors (residuals)
residuals = y_test.values - y_pred

plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=25, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribution of Prediction Errors (Residuals)')
plt.xlabel('Error (Actual - Predicted)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## 9. Save the Model and Scaler

We save the trained model and the scaler so they can be reused later without retraining.

In [ ]:
import pickle

# Save the model
model.save('models/sleep_efficiency_model.keras')
print('Model saved!')

# Save the scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Scaler saved!')

---
## 10. Summary and Conclusions

In this project we built an Artificial Neural Network to predict sleep efficiency based on lifestyle factors.

**What we found in EDA:**
- *[Write your observations here after running the notebook – what patterns did you see?]*
- *[For example: Does caffeine clearly affect sleep efficiency? What about alcohol?]*

**Model performance:**
- *[Fill in your R², MAE and RMSE values here after training]*
- *[Comment on whether the model is good – is there overfitting?]*

**Can the model be improved?**
- Try adding more layers or neurons
- Try more epochs
- Try including the sleep phase columns (REM, deep, light)

**Business value:**
A health app or insurance company could use this model to give personalized sleep recommendations based on a user's lifestyle. By adjusting factors like caffeine intake or exercise frequency, the model can show how sleep efficiency would change – helping users make better decisions for their health.